# European Soccer Data - Descriptive Statistics

This notebook computes basic descriptive statistics for the European soccer match data across all leagues:
- Premier League
- Serie A 
- Bundesliga
- La Liga

## Statistics Computed:
- Number of observations/matches per league
- Win/draw percentages
- Goal statistics (mean/std)
- Betting odds statistics (mean/std)

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

# Set display options
pd.set_option('display.max_columns', None)
pd.set_option('display.precision', 3)

# Set plotting style
plt.style.use('default')
sns.set_palette("husl")

ModuleNotFoundError: No module named 'pandas'

## Load Data

In [ ]:
# Define data paths
data_dir = Path('../data/processed')

# Load all league datasets
leagues = {
    'Premier League': pd.read_csv(data_dir / 'premier_league.csv'),
    'Serie A': pd.read_csv(data_dir / 'serie_a.csv'),
    'Bundesliga': pd.read_csv(data_dir / 'bundesliga.csv'),
    'La Liga': pd.read_csv(data_dir / 'la_liga.csv')
}

# Display basic info about each dataset
print("Dataset Overview:")
print("=" * 50)
for name, df in leagues.items():
    print(f"{name}: {len(df):,} matches")
    print(f"  Seasons: {df['season'].min()}-{df['season'].max()}")
    print(f"  Date range: {df['date'].min()} to {df['date'].max()}")
    print()

## 1. Basic Match Statistics

In [ ]:
# Create summary statistics table
summary_stats = []

for league_name, df in leagues.items():
    # Filter out any rows with missing data
    df_clean = df.dropna(subset=['hometeamresult'])
    
    # Calculate result percentages
    result_counts = df_clean['hometeamresult'].value_counts()
    total_matches = len(df_clean)
    
    home_wins = result_counts.get('win', 0) / total_matches * 100
    draws = result_counts.get('draw', 0) / total_matches * 100
    away_wins = result_counts.get('loss', 0) / total_matches * 100
    
    summary_stats.append({
        'League': league_name,
        'Total Matches': total_matches,
        'Home Wins (%)': home_wins,
        'Draws (%)': draws,
        'Away Wins (%)': away_wins
    })

# Create DataFrame and display
summary_df = pd.DataFrame(summary_stats)
print("Match Results Summary")
print("=" * 60)
print(summary_df.to_string(index=False, float_format='%.1f'))

# Calculate overall statistics
total_matches = summary_df['Total Matches'].sum()
overall_home = (summary_df['Home Wins (%)'] * summary_df['Total Matches']).sum() / total_matches
overall_draw = (summary_df['Draws (%)'] * summary_df['Total Matches']).sum() / total_matches
overall_away = (summary_df['Away Wins (%)'] * summary_df['Total Matches']).sum() / total_matches

print(f"\nOverall Statistics (All Leagues):")
print(f"Total Matches: {total_matches:,}")
print(f"Home Wins: {overall_home:.1f}%")
print(f"Draws: {overall_draw:.1f}%")
print(f"Away Wins: {overall_away:.1f}%")

## 2. Goal Statistics

In [ ]:
# Calculate goal statistics
goal_stats = []

for league_name, df in leagues.items():
    # Filter out any rows with missing goal data
    df_clean = df.dropna(subset=['hometeamgoals', 'awayteamgoals'])
    
    # Calculate total goals per match
    df_clean['total_goals'] = df_clean['hometeamgoals'] + df_clean['awayteamgoals']
    
    goal_stats.append({
        'League': league_name,
        'Avg Home Goals': df_clean['hometeamgoals'].mean(),
        'Std Home Goals': df_clean['hometeamgoals'].std(),
        'Avg Away Goals': df_clean['awayteamgoals'].mean(),
        'Std Away Goals': df_clean['awayteamgoals'].std(),
        'Avg Total Goals': df_clean['total_goals'].mean(),
        'Std Total Goals': df_clean['total_goals'].std()
    })

goal_stats_df = pd.DataFrame(goal_stats)
print("Goal Statistics by League")
print("=" * 80)
print(goal_stats_df.to_string(index=False, float_format='%.2f'))

## 3. Betting Odds Statistics

In [ ]:
# Calculate betting odds statistics
odds_stats = []

for league_name, df in leagues.items():
    # Filter out any rows with missing odds data
    odds_cols = ['OddHome', 'OddDraw', 'OddAway']
    df_clean = df.dropna(subset=odds_cols)
    
    if len(df_clean) > 0:
        odds_stats.append({
            'League': league_name,
            'Matches with Odds': len(df_clean),
            'Avg Home Odds': df_clean['OddHome'].mean(),
            'Std Home Odds': df_clean['OddHome'].std(),
            'Avg Draw Odds': df_clean['OddDraw'].mean(),
            'Std Draw Odds': df_clean['OddDraw'].std(),
            'Avg Away Odds': df_clean['OddAway'].mean(),
            'Std Away Odds': df_clean['OddAway'].std()
        })
    else:
        odds_stats.append({
            'League': league_name,
            'Matches with Odds': 0,
            'Avg Home Odds': np.nan,
            'Std Home Odds': np.nan,
            'Avg Draw Odds': np.nan,
            'Std Draw Odds': np.nan,
            'Avg Away Odds': np.nan,
            'Std Away Odds': np.nan
        })

odds_stats_df = pd.DataFrame(odds_stats)
print("Betting Odds Statistics by League")
print("=" * 80)
print(odds_stats_df.to_string(index=False, float_format='%.2f'))

## 4. Visualizations

In [ ]:
# Create visualization of match results
fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(2, 2, figsize=(15, 12))

# 1. Match results by league
result_data = summary_df.set_index('League')[['Home Wins (%)', 'Draws (%)', 'Away Wins (%)']]
result_data.plot(kind='bar', ax=ax1, width=0.8)
ax1.set_title('Match Results by League', fontsize=14, fontweight='bold')
ax1.set_ylabel('Percentage')
ax1.legend(title='Result Type')
ax1.tick_params(axis='x', rotation=45)

# 2. Average goals by league
goal_data = goal_stats_df.set_index('League')[['Avg Home Goals', 'Avg Away Goals']]
goal_data.plot(kind='bar', ax=ax2, width=0.8)
ax2.set_title('Average Goals by League', fontsize=14, fontweight='bold')
ax2.set_ylabel('Average Goals')
ax2.legend(title='Team Type')
ax2.tick_params(axis='x', rotation=45)

# 3. Total goals distribution (combined data)
all_total_goals = []
for df in leagues.values():
    df_clean = df.dropna(subset=['hometeamgoals', 'awayteamgoals'])
    all_total_goals.extend(df_clean['hometeamgoals'] + df_clean['awayteamgoals'])

ax3.hist(all_total_goals, bins=range(0, 12), alpha=0.7, edgecolor='black')
ax3.set_title('Distribution of Total Goals per Match', fontsize=14, fontweight='bold')
ax3.set_xlabel('Total Goals')
ax3.set_ylabel('Frequency')
ax3.grid(axis='y', alpha=0.3)

# 4. Average betting odds by league
if not odds_stats_df['Avg Home Odds'].isna().all():
    odds_data = odds_stats_df.set_index('League')[['Avg Home Odds', 'Avg Draw Odds', 'Avg Away Odds']]
    odds_data.plot(kind='bar', ax=ax4, width=0.8)
    ax4.set_title('Average Betting Odds by League', fontsize=14, fontweight='bold')
    ax4.set_ylabel('Average Odds')
    ax4.legend(title='Bet Type')
    ax4.tick_params(axis='x', rotation=45)
else:
    ax4.text(0.5, 0.5, 'No odds data available', ha='center', va='center', transform=ax4.transAxes)
    ax4.set_title('Average Betting Odds by League', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.show()

## 5. Combined Summary Statistics

In [ ]:
# Create a comprehensive summary table
print("\n" + "=" * 80)
print("COMPREHENSIVE SUMMARY - EUROPEAN SOCCER DATA")
print("=" * 80)

# Combine all data for overall statistics
all_data = pd.concat(leagues.values(), ignore_index=True)
all_data_clean = all_data.dropna(subset=['hometeamresult', 'hometeamgoals', 'awayteamgoals'])

print(f"\nOVERALL STATISTICS:")
print(f"Total matches analyzed: {len(all_data_clean):,}")
print(f"Leagues covered: {len(leagues)}")
print(f"Season range: {all_data['season'].min()}-{all_data['season'].max()}")

# Result percentages
result_pct = all_data_clean['hometeamresult'].value_counts(normalize=True) * 100
print(f"\nMATCH RESULTS:")
print(f"Home wins: {result_pct.get('win', 0):.1f}%")
print(f"Draws: {result_pct.get('draw', 0):.1f}%")
print(f"Away wins: {result_pct.get('loss', 0):.1f}%")

# Goal statistics
print(f"\nGOAL STATISTICS:")
print(f"Average home goals: {all_data_clean['hometeamgoals'].mean():.2f} ± {all_data_clean['hometeamgoals'].std():.2f}")
print(f"Average away goals: {all_data_clean['awayteamgoals'].mean():.2f} ± {all_data_clean['awayteamgoals'].std():.2f}")
total_goals = all_data_clean['hometeamgoals'] + all_data_clean['awayteamgoals']
print(f"Average total goals: {total_goals.mean():.2f} ± {total_goals.std():.2f}")

# Odds statistics (if available)
odds_data = all_data.dropna(subset=['OddHome', 'OddDraw', 'OddAway'])
if len(odds_data) > 0:
    print(f"\nBETTING ODDS STATISTICS:")
    print(f"Matches with odds data: {len(odds_data):,} ({len(odds_data)/len(all_data)*100:.1f}%)")
    print(f"Average home odds: {odds_data['OddHome'].mean():.2f} ± {odds_data['OddHome'].std():.2f}")
    print(f"Average draw odds: {odds_data['OddDraw'].mean():.2f} ± {odds_data['OddDraw'].std():.2f}")
    print(f"Average away odds: {odds_data['OddAway'].mean():.2f} ± {odds_data['OddAway'].std():.2f}")
else:
    print(f"\nBETTING ODDS: No odds data available")

print("\n" + "=" * 80)